# 프로젝트 Notebook 05. 설명 가능한 입지 추천

목표: 기준 방향을 통일하고 정규화·가중합 점수를 계산하여 추천 근거와 민감도를 제시한다.

$$S_i=\sum_j w_jz_{ij},\qquad \sum_jw_j=1$$

In [ ]:
from pathlib import Path
import sys, subprocess

REPO_URL = "https://github.com/niko2204/bigdataservice.git"
if "google.colab" in sys.modules:
    ROOT = Path("/content/bigdataservice")
    if not ROOT.exists():
        subprocess.run(["git", "clone", "-q", REPO_URL, str(ROOT)], check=True)
else:
    candidates = [Path.cwd(), Path.cwd().parent, Path.cwd().parent.parent]
    ROOT = next((p.resolve() for p in candidates if (p / "data").exists()), None)
    if ROOT is None:
        raise FileNotFoundError("bigdataservice 저장소 안에서 실행하세요.")
print("저장소:", ROOT)

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
df = pd.read_csv(ROOT / "data/sample/location_features.csv")

## 1. 기준과 방향

잠재고객·유동인구·접근성은 클수록 좋고 경쟁·비용은 작을수록 좋다고 가정한다.
이 방향은 목적에 따른 선택이며 객관적 진실이 아니다.

In [ ]:
raw_features = pd.DataFrame({
    "잠재고객": df["20대인구"],
    "유동인구": df["유동인구"],
    "경쟁완화": -df["카페수"],
    "접근성": df["주차장수"],
    "비용효율": -df["평균거래가"],
})
display(raw_features.head())

## 2. Min-Max 정규화

모든 값이 같은 열은 범위가 0이므로 중립값 0.5를 부여한다.

In [ ]:
def minmax(series):
    span = series.max() - series.min()
    if span == 0:
        return pd.Series(0.5, index=series.index)
    return (series - series.min()) / span

normalized = raw_features.apply(minmax)
assert normalized.min().min() >= 0
assert normalized.max().max() <= 1
display(normalized.round(3))

## 3. 가중합 점수와 순위

In [ ]:
weights = pd.Series({
    "잠재고객": .25, "유동인구": .25, "경쟁완화": .20,
    "접근성": .15, "비용효율": .15,
})
assert np.isclose(weights.sum(), 1)

result = df[["행정동명"]].copy()
result["적합도"] = normalized.mul(weights, axis=1).sum(axis=1) * 100
result["순위"] = result["적합도"].rank(method="min", ascending=False).astype(int)
result = result.sort_values("순위")
display(result)

## 4. 지표별 점수 기여

총점뿐 아니라 지표별 기여도를 보여야 사용자가 추천을 검증할 수 있다.

In [ ]:
contribution = normalized.mul(weights, axis=1) * 100
contribution.insert(0, "행정동명", df["행정동명"])
top_index = result.index[0]
top_contribution = contribution.loc[top_index].drop("행정동명").sort_values(ascending=False)
print("1위:", df.loc[top_index, "행정동명"])
display(top_contribution.rename("점수기여").to_frame())

## 5. 민감도 분석

선호가 달라져도 상위 지역이 유지되는지 세 시나리오를 비교한다.

In [ ]:
scenarios = {
    "균형": [.25, .25, .20, .15, .15],
    "유동중심": [.15, .45, .15, .10, .15],
    "저비용": [.15, .15, .15, .10, .45],
}
ranking = pd.DataFrame({"행정동명": df["행정동명"]})
for name, values in scenarios.items():
    w = pd.Series(values, index=normalized.columns)
    score = normalized.mul(w, axis=1).sum(axis=1)
    ranking[name] = score.rank(method="min", ascending=False).astype(int)
ranking["순위범위"] = ranking[list(scenarios)].max(axis=1) - ranking[list(scenarios)].min(axis=1)
display(ranking.sort_values("균형"))

순위범위가 크면 하나의 절대적 추천 대신 조건별 후보를 제시한다.

## 6. 독립 연습

1. 업종을 음식점·편의점으로 바꾸어 경쟁 기준을 변경한다.
2. 30대 고객 시나리오를 추가한다.
3. Min-Max 대신 z점수 또는 순위점수를 적용해 상위 5개를 비교한다.
4. 각 가중치를 ±10% 변화시키는 민감도 분석을 자동화한다.
5. 추천 1위의 강점·약점과 실제 적용 전 필요한 데이터를 작성한다.

In [ ]:
# TODO: 업종·고객·가중치를 인자로 받는 함수
def recommend(data, business_type, target_age, weights):
    pass